In [10]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [18]:
import jax
import jax.numpy as jnp
from pathlib import Path
from state_estimation.kf_utils import load_custom_dataset
from state_estimation.run_estimation import run_estimation
from state_estimation.kf_utils import build_noise_vector

sim_num = 3
dataset_num = 3
data = load_custom_dataset(dataset_path=Path.cwd().parent / f"custom_datasets/quad_mass_dataset_run{dataset_num}.npz", sim_num=sim_num)

Q = build_noise_vector([[0.0001], [0.01], [0.01], [1]], sizes=[3,3,3,12])
R = build_noise_vector([[0.01], [0.0001], [1]], sizes=[3,3,12])

new_dict = {k: v for k, v in data.items() if k != 'base_acc'}
new_dict = {k: v for k, v in new_dict.items() if k != 'contact_forces'}
new_dict = {k: v for k, v in new_dict.items() if k != 'contact_states'}

model_name = "epochs_1_quad_mass_dataset_run8_input_values_joint_seed_0_lstm5x10_span0.5-stride4"
cadelan_path = Path(f"trained_models/CaDeLaN/{model_name}")

result = run_estimation(dt=data["dt"][0], 
                        data=new_dict,
                        Q=Q, 
                        R=R,
                        base_acc_source="external",
                        model_name="aliengo",
                        include_ang_vel=True,
                        include_contact_force=True,
                        use_jax=True,
                        cadelan_path=cadelan_path)

Data loaded from /home/jad/Uni/Master/semester-2/RL-IP/mpx/custom_datasets/quad_mass_dataset_run3.npz.

═══════════════════ Missing inputs - Estimating internally ═══════════════════
  • contact_state - Estimated based on momentum
  • contact_forces - Estimated with joint torque
  • base_acc - Estimated with dynamics based on Rigid-Body-Dynamics (external)
══════════════════════════════════════════════════════════════════════════════
Running with JAX
Loading CaDeLaN model
Preparation setup finished after 0.04s
JAX Lax Scan finished after 3.77s


In [19]:
from state_estimation.evaluate import evaluate_estimation
evaluate_estimation(result, data, plot_results=False)


═══════════════════════ RMSE — State Estimation ════════════════════════
Quantity             Unit             x          y          z        ‖·‖
════════════════════════════════════════════════════════════════════════
Position             m           0.1122     0.0311     0.4013     0.4179
────────────────────────────────────────────────────────────────────────
Linear Velocity      m/s         0.1123     0.0334     0.1085     0.1596
────────────────────────────────────────────────────────────────────────
Angular Velocity     rad/s       0.0011     0.0008     0.0002     0.0013
────────────────────────────────────────────────────────────────────────
Contact Force FL     N           8.2075    10.4955    43.1137    45.1255
────────────────────────────────────────────────────────────────────────
Contact Force FR     N           9.8487     7.1098    42.8808    44.5680
────────────────────────────────────────────────────────────────────────
Contact Force RL     N           9.7893     6.8693

In [75]:
from pathlib import Path
from state_estimation.kf_utils import load_custom_dataset
from state_estimation.run_estimation import run_estimation
from state_estimation.kf_utils import build_noise_vector
from state_estimation.evaluate import evaluate_estimation

sim_num = 3
dataset_num = 3
data = load_custom_dataset(dataset_path=Path.cwd().parent / f"custom_datasets/quad_dataset_run{dataset_num}.npz", sim_num=sim_num)
Q = build_noise_vector([[0.0001], [0.000001], [0.01], [0.1]], sizes=[3,3,3,12])
R = build_noise_vector([[0.01], [0.0001], [0.1]], sizes=[3,3,12])

Data loaded from /home/jad/Uni/Master/semester-2/RL-IP/mpx/custom_datasets/quad_dataset_run3.npz.


## Using ground truth acceleration

In [47]:
Q = build_noise_vector([[0.0001], [0.01], [0.01], [1]], sizes=[3,3,3,12])
R = build_noise_vector([[0.01], [0.0001], [1]], sizes=[3,3,12])

result = run_estimation(dt=data["dt"][0], 
                        data=data,
                        Q=Q, 
                        R=R,
                        base_acc_source="internal",
                        model_name="aliengo",
                        include_ang_vel=True,
                        include_contact_force=True)

Running state estimation: 100%|██████████| 5000/5000 [00:02<00:00, 2088.14it/s]


In [ ]:
evaluate_estimation(result, data, plot_results=False)

## Estimating acceleration

In [51]:
Q = build_noise_vector([[0.0001], [0.01], [0.01], [1]], sizes=[3,3,3,12])
R = build_noise_vector([[0.01], [0.0001], [1]], sizes=[3,3,12])
new_dict = {k: v for k, v in data.items() if k != 'base_acc'}

result = run_estimation(dt=data["dt"][0], 
                        data=new_dict,
                        Q=Q, 
                        R=R,
                        base_acc_source="external",
                        model_name="aliengo",
                        include_ang_vel=True,
                        include_contact_force=True)


═══════════════════ Missing inputs - Estimating internally ═══════════════════
  • base_acc - Estimated with dynamics based on Rigid-Body-Dynamics (external)
══════════════════════════════════════════════════════════════════════════════


Running state estimation: 100%|██████████| 5000/5000 [00:02<00:00, 1955.58it/s]


In [52]:
evaluate_estimation(result, data, plot_results=False)


═══════════════════════ RMSE — State Estimation ════════════════════════
Quantity             Unit             x          y          z        ‖·‖
════════════════════════════════════════════════════════════════════════
Position             m           0.2678     0.0192     0.9679     1.0044
────────────────────────────────────────────────────────────────────────
Linear Velocity      m/s         0.0635     0.0329     0.1624     0.1774
────────────────────────────────────────────────────────────────────────
Angular Velocity     rad/s       0.0006     0.0004     0.0001     0.0007
────────────────────────────────────────────────────────────────────────
Contact Force FL     N           1.4621     0.9548     4.5369     4.8614
────────────────────────────────────────────────────────────────────────
Contact Force FR     N           1.5628     0.8492     4.3998     4.7457
────────────────────────────────────────────────────────────────────────
Contact Force RL     N           1.4280     0.9167

## Estimating contact force

In [76]:
new_dict = {k: v for k, v in data.items() if k != 'base_acc'}
new_dict = {k: v for k, v in new_dict.items() if k != 'contact_forces'}

result = run_estimation(dt=data["dt"][0], 
                        data=new_dict,
                        Q=Q, 
                        R=R,
                        base_acc_source="internal",
                        model_name="aliengo",
                        include_ang_vel=True,
                        include_contact_force=True)


═══════════════════ Missing inputs - Estimating internally ═══════════════════
  • contact_forces - Estimated with joint torque
  • base_acc - Estimated with dynamics based on Rigid-Body-Dynamics (internal)
══════════════════════════════════════════════════════════════════════════════


Running state estimation: 100%|██████████| 5000/5000 [00:02<00:00, 1765.45it/s]


In [70]:
evaluate_estimation(result, data, plot_results=False)


═══════════════════════ RMSE — State Estimation ════════════════════════
Quantity             Unit             x          y          z        ‖·‖
════════════════════════════════════════════════════════════════════════
Position             m           0.2678     0.0192     0.9679     1.0044
────────────────────────────────────────────────────────────────────────
Linear Velocity      m/s         0.0631     0.0331     0.1675     0.1820
────────────────────────────────────────────────────────────────────────
Angular Velocity     rad/s       0.0008     0.0004     0.0002     0.0009
────────────────────────────────────────────────────────────────────────
Contact Force FL     N           7.5508     4.9403    22.0513    23.8260
────────────────────────────────────────────────────────────────────────
Contact Force FR     N           6.7048     4.9840    19.6223    21.3268
────────────────────────────────────────────────────────────────────────
Contact Force RL     N           6.8090     4.4762

## Estimating contact state

In [ ]:
Q = build_noise_vector([[0.0001], [0.01], [0.01], [1]], sizes=[3,3,3,12])
R = build_noise_vector([[0.01], [0.0001], [1]], sizes=[3,3,12])
# new_dict = {k: v for k, v in data.items() if k != 'base_acc'}
# new_dict = {k: v for k, v in new_dict.items() if k != 'contact_forces'}
new_dict = {k: v for k, v in new_dict.items() if k != 'contact_states'}

result = run_estimation(dt=data["dt"][0], 
                        data=new_dict,
                        Q=Q, 
                        R=R,
                        base_acc_source="internal",
                        model_name="aliengo",
                        include_ang_vel=True,
                        include_contact_force=True)

In [ ]:
evaluate_estimation(result, data, plot_results=False)